In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install PyMuPDF pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 47.3 MB/s eta 0:00:00


In [3]:
import os
import re
import pymupdf
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
drive_path = "/content/drive/MyDrive/research paper project/RP pdf"

pdf_files = sorted([
    file for file in os.listdir(drive_path)
    if file.lower().endswith(".pdf")
])

print("PDF files found:")

for file in pdf_files:
    print(file)

print("\nTotal PDFs:", len(pdf_files))

PDF files found:
RP1.pdf
RP2.pdf
RP3.pdf
RP4.pdf
RP5.pdf

Total PDFs: 5


In [5]:
all_papers = {}
paper_pages = {}

for file_name in pdf_files:

    file_path = os.path.join(drive_path, file_name)

    with pymupdf.open(file_path) as pdf:

        page_count = len(pdf)

        text = ""

        for page in pdf:
            text += page.get_text() + "\n"

    # Clean unnecessary line breaks and spacing
    text = re.sub(r'-\s*\n\s*', '', text)
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    all_papers[file_name] = text
    paper_pages[file_name] = page_count

    print(f"✅ {file_name} extracted")
    print("Pages:", page_count)
    print("Words:", len(text.split()))
    print("-" * 40)

print("Total papers extracted:", len(all_papers))

✅ RP1.pdf extracted
Pages: 5
Words: 3938
----------------------------------------
✅ RP2.pdf extracted
Pages: 16
Words: 8712
----------------------------------------
✅ RP3.pdf extracted
Pages: 29
Words: 13061
----------------------------------------
✅ RP4.pdf extracted
Pages: 11
Words: 4466
----------------------------------------
✅ RP5.pdf extracted
Pages: 32
Words: 10899
----------------------------------------
Total papers extracted: 5


In [6]:
for paper_name, paper_text in all_papers.items():

    print("\n", "=" * 60)
    print("PAPER:", paper_name)
    print("=" * 60)

    print(paper_text[:1200])

    print("\nTotal characters:", len(paper_text))


PAPER: RP1.pdf
Epidemiological Causal Graph Identification: Challenges,
Identifiability and Algorithms
Sambit Mishra∗, Yingying Wang†, Christine K. Johnson†, and Urbashi Mitra∗
∗University of Southern California, † University of California, Davis
Abstract—Causal discovery from observational data is fundamental to statistics and machine learning, yet determining
causal direction without interventions necessitates structural
assumptions. Existing identifiability research primarily focuses
on continuous variables under additive noise models, often
neglecting mixed datasets containing ordinal scales, counts,
and continuous measurements. This paper investigates causal
discovery in Directed Acyclic Graphs (DAGs) where nodes follow
either an ordinal distribution (via an ordered logit model) or a
regular one-parameter exponential family distribution. We prove
that the edge direction between an ordinal and an exponential
family node is distributionally identifiable for generic parameter
values

In [7]:
def extract_abstract(text):

    pattern = re.compile(
        r'\babstract\b\s*[:.]?\s*'
        r'(.*?)'
        r'(?=\b(?:keywords?|index terms|'
        r'1\.?\s+introduction|'
        r'i\.?\s+introduction)\b)',
        re.IGNORECASE | re.DOTALL
    )

    match = pattern.search(text)

    if match:
        abstract = match.group(1).strip()
        return re.sub(r'\s+', ' ', abstract)

    return "Abstract not detected"

In [8]:
def extract_keywords(text):

    pattern = re.compile(
        r'\b(?:keywords?|index terms)\b\s*[:\-]?\s*'
        r'(.*?)'
        r'(?=\n\s*\n|\n\s*(?:1\.?\s+introduction|'
        r'i\.?\s+introduction)\b)',
        re.IGNORECASE | re.DOTALL
    )

    match = pattern.search(text)

    if match:
        keywords = match.group(1)
        keywords = re.sub(r'\s+', ' ', keywords)
        keywords = keywords.strip(" :-–—")

        return keywords

    return "Keywords not detected"

In [9]:
def extract_year(text):

    # Look in the first 2500 characters
    first_part = text[:2500]

    years = re.findall(
        r'\b(?:19|20)\d{2}\b',
        first_part
    )

    if years:
        return years[0]

    return "Year not detected"

In [10]:
def extract_title(text):

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    # Use the first few lines as a title candidate
    title_lines = []

    for line in lines[:8]:

        # Stop if we reach the abstract heading
        if re.match(r'abstract\b', line, re.IGNORECASE):
            break

        # Ignore very short lines
        if len(line) > 3:
            title_lines.append(line)

    title = " ".join(title_lines)

    return re.sub(r'\s+', ' ', title).strip()

In [11]:
paper_records = []

for paper_name, paper_text in all_papers.items():

    record = {
        "Paper_ID": paper_name.replace(".pdf", ""),
        "File_Name": paper_name,
        "Title": extract_title(paper_text),
        "Publication_Year": extract_year(paper_text),
        "Abstract": extract_abstract(paper_text),
        "Keywords": extract_keywords(paper_text),
        "Pages": paper_pages[paper_name],
        "Word_Count": len(paper_text.split())
    }

    paper_records.append(record)

papers_df = pd.DataFrame(paper_records)

papers_df

,Paper_ID,File_Name,Title,Publication_Year,Abstract,Keywords,Pages,Word_Count
0,RP1,RP1.pdf,Epidemiological Causal Graph Identification: C...,Year not detected,—Causal discovery from observational data is f...,"Causal Discovery, Directed Acyclic Graphs, Ide...",5,3938
1,RP2,RP2.pdf,SCORE CENTERING STABILIZES OFF-POLICY REINFORC...,2025,Reinforcement learning (RL) of large language ...,Keywords not detected,16,8712
2,RP3,RP3.pdf,Paint-Anything: Unified Any-Color Control for ...,2026,Professional design requires any-color control...,Keywords not detected,29,13061
3,RP4,RP4.pdf,Towards AI-enhanced control: a numerical techn...,Year not detected,The paper presents a numerical approach for th...,"Trajectory Smoothing, Parallel Robot, Minimall...",11,4466
4,RP5,RP5.pdf,Detecting Deceptive Recruitment: A Signal-theo...,Year not detected,Deceptive online job advertisements have emerg...,"density, and visa sponsorship mentions ranking...",32,10899


In [12]:
def generate_summary(text, num_sentences=3):

    # Split text into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)

    sentences = [
        s.strip()
        for s in sentences
        if len(s.split()) >= 5
    ]

    if len(sentences) == 0:
        return "Summary unavailable"

    if len(sentences) <= num_sentences:
        return " ".join(sentences)

    # Convert sentences into TF-IDF vectors
    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    try:
        tfidf_matrix = vectorizer.fit_transform(sentences)

        # Calculate sentence similarity
        similarity_matrix = cosine_similarity(tfidf_matrix)

        # Rank sentences by overall similarity
        scores = similarity_matrix.sum(axis=1)

        # Select top-ranked sentences
        top_indices = scores.argsort()[-num_sentences:]

        # Preserve original sentence order
        top_indices = sorted(top_indices)

        summary = " ".join(
            sentences[i] for i in top_indices
        )

        return summary

    except ValueError:
        return "Summary unavailable"

In [13]:
summaries = []

for paper_name, paper_text in all_papers.items():

    abstract = extract_abstract(paper_text)

    if abstract == "Abstract not detected":
        abstract = paper_text[:3000]

    summary = generate_summary(
        abstract,
        num_sentences=3
    )

    summaries.append(summary)

papers_df["Summary"] = summaries

papers_df[["Paper_ID", "Summary"]]

,Paper_ID,Summary
0,RP1,This paper investigates causal discovery in Di...
1,RP2,We derive an additive “score centering” correc...
2,RP3,Professional design requires any-color control...
3,RP4,The approach is tailored for real-time master-...
4,RP5,Deceptive online job advertisements have emerg...


In [14]:
abstracts = []

for paper_name, paper_text in all_papers.items():

    abstract = extract_abstract(paper_text)

    if abstract == "Abstract not detected":
        abstract = paper_text[:3000]

    abstracts.append(abstract)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=1000,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(abstracts)

terms = vectorizer.get_feature_names_out()

tfidf_keywords = []

for i in range(len(abstracts)):

    scores = tfidf_matrix[i].toarray().flatten()

    top_indices = scores.argsort()[-10:][::-1]

    top_terms = [
        terms[index]
        for index in top_indices
        if scores[index] > 0
    ]

    tfidf_keywords.append(", ".join(top_terms))

papers_df["TFIDF_Keywords"] = tfidf_keywords

papers_df[["Paper_ID", "TFIDF_Keywords"]]

,Paper_ID,TFIDF_Keywords
0,RP1,"ordinal, causal, continuous, exponential, expo..."
1,RP2,"training, correction, score centering, importa..."
2,RP3,"color, acbench, hex, object, editing, generati..."
3,RP4,"approach, end effector, space, command, end, t..."
4,RP5,"detection, deceptive, legitimate, systematic, ..."


In [15]:
output_path = "/content/drive/MyDrive/research paper project"

os.makedirs(output_path, exist_ok=True)

csv_path = os.path.join(
    output_path,
    "research_paper_insights.csv"
)

papers_df.to_csv(csv_path, index=False)

print("Dataset saved successfully!")
print(csv_path)

Dataset saved successfully!
/content/drive/MyDrive/research paper project/research_paper_insights.csv


In [16]:
display(
    papers_df[
        [
            "Paper_ID",
            "Title",
            "Publication_Year",
            "Keywords",
            "Summary",
            "TFIDF_Keywords"
        ]
    ]
)

,Paper_ID,Title,Publication_Year,Keywords,Summary,TFIDF_Keywords
0,RP1,Epidemiological Causal Graph Identification: C...,Year not detected,"Causal Discovery, Directed Acyclic Graphs, Ide...",This paper investigates causal discovery in Di...,"ordinal, causal, continuous, exponential, expo..."
1,RP2,SCORE CENTERING STABILIZES OFF-POLICY REINFORC...,2025,Keywords not detected,We derive an additive “score centering” correc...,"training, correction, score centering, importa..."
2,RP3,Paint-Anything: Unified Any-Color Control for ...,2026,Keywords not detected,Professional design requires any-color control...,"color, acbench, hex, object, editing, generati..."
3,RP4,Towards AI-enhanced control: a numerical techn...,Year not detected,"Trajectory Smoothing, Parallel Robot, Minimall...",The approach is tailored for real-time master-...,"approach, end effector, space, command, end, t..."
4,RP5,Detecting Deceptive Recruitment: A Signal-theo...,Year not detected,"density, and visa sponsorship mentions ranking...",Deceptive online job advertisements have emerg...,"detection, deceptive, legitimate, systematic, ..."


In [17]:
# Check the quality of extracted research paper data

print("TOTAL PAPERS:", len(papers_df))

print("\nMISSING VALUES:")
print(papers_df.isnull().sum())

print("\nDUPLICATE PAPERS:")
print(papers_df["Paper_ID"].duplicated().sum())

print("\nPAPER TITLES:")
for title in papers_df["Title"]:
    print("-", title)

print("\nABSTRACTS:")
for i, abstract in enumerate(papers_df["Abstract"], start=1):
    print(f"\nPaper {i}:")
    print(abstract[:300])

TOTAL PAPERS: 5

MISSING VALUES:
Paper_ID            0
File_Name           0
Title               0
Publication_Year    0
Abstract            0
Keywords            0
Pages               0
Word_Count          0
Summary             0
TFIDF_Keywords      0
dtype: int64

DUPLICATE PAPERS:
0

PAPER TITLES:
- Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms Sambit Mishra∗, Yingying Wang†, Christine K. Johnson†, and Urbashi Mitra∗ ∗University of Southern California, † University of California, Davis
- SCORE CENTERING STABILIZES OFF-POLICY REINFORCEMENT LEARNING Martin Marek & Max Ryabinin Together AI
- Paint-Anything: Unified Any-Color Control for Image Generation and Editing Ji Xie1,2, Dewei Zhou1,2, Xinyu Huang1, Zhennan Chen1,3, Xun Wang1 1ByteDance Seed, 2Zhejiang University, 3Nanjing University
- Towards AI-enhanced control: a numerical technique for trajectory smoothing of a parallel robot for pancreatic surgery Iosif Birlescu1[0000-0002-4026-2318],

In [18]:
# Correct titles for the 5 research papers

title_corrections = {
    "RP1": "Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms",

    "RP2": "Score Centering Stabilizes Off-Policy Reinforcement Learning",

    "RP3": "Paint-Anything: Unified Any-Color Control for Image Generation and Editing",

    "RP4": "Towards AI-enhanced control: a numerical technique for trajectory smoothing of a parallel robot for pancreatic surgery",

    "RP5": "Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identification of Labour Exploitation"
}

# Update titles in the dataset
papers_df["Title"] = papers_df["Paper_ID"].map(title_corrections)

# Display corrected titles
for i, row in papers_df.iterrows():
    print(f"{row['Paper_ID']}: {row['Title']}")
    print()

RP1: Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms

RP2: Score Centering Stabilizes Off-Policy Reinforcement Learning

RP3: Paint-Anything: Unified Any-Color Control for Image Generation and Editing

RP4: Towards AI-enhanced control: a numerical technique for trajectory smoothing of a parallel robot for pancreatic surgery

RP5: Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identification of Labour Exploitation



In [19]:
# Clean and standardise the paper titles

clean_titles = {
    "RP1.pdf": "Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms",

    "RP2.pdf": "Score Centering Stabilizes Off-Policy Reinforcement Learning",

    "RP3.pdf": "Paint-Anything: Unified Any-Color Control for Image Generation and Editing",

    "RP4.pdf": "Towards AI-enhanced control: A numerical technique for trajectory smoothing of a parallel robot for pancreatic surgery",

    "RP5.pdf": "Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identification of Labour Exploitation"
}

# Apply clean titles to your dataframe
# If your dataframe is named df instead of papers_df,
# replace papers_df with df in the next line.

papers_df["Title"] = papers_df["File_Name"].map(clean_titles).fillna(papers_df["Title"])

print("CLEANED PAPER TITLES:\n")

for i, title in enumerate(papers_df["Title"], start=1):
    print(f"RP{i}: {title}")

CLEANED PAPER TITLES:

RP1: Epidemiological Causal Graph Identification: Challenges, Identifiability and Algorithms
RP2: Score Centering Stabilizes Off-Policy Reinforcement Learning
RP3: Paint-Anything: Unified Any-Color Control for Image Generation and Editing
RP4: Towards AI-enhanced control: A numerical technique for trajectory smoothing of a parallel robot for pancreatic surgery
RP5: Detecting Deceptive Recruitment: A Signal-theoretic Machine Learning Framework for Early Identification of Labour Exploitation
